# IN6227 Assignment 1 — Variant 1
## Part 2: Data Cleaning

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
RANDOM_STATE = 42
TARGET = "label"

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

num_cols = train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in train.columns if c not in num_cols + [TARGET]]

# Keep a copy of the raw data so we can compare before / after at the end
train_raw, test_raw = train.copy(), test.copy()
print("train:", train.shape, " test:", test.shape)

train: (31112, 16)  test: (13334, 16)


### 2.1 Drop rows with a missing label

In [3]:
print("Missing labels -> train:", train[TARGET].isnull().sum(), "| test:", test[TARGET].isnull().sum())

train = train.dropna(subset=[TARGET]).reset_index(drop=True)
test = test.dropna(subset=[TARGET]).reset_index(drop=True)

print("After dropping -> train:", train.shape, " test:", test.shape)
print("Label values:", sorted(train[TARGET].unique()), sorted(test[TARGET].unique()))

Missing labels -> train: 3 | test: 1
After dropping -> train: (31109, 16)  test: (13333, 16)
Label values: ['no', 'yes'] ['no', 'yes']


### 2.2 Group rare categories into `Other`

In [7]:
RARE_THRESHOLD = 0.01   # 1% of training rows

rare_levels = {}
for c in cat_cols:
    share = train[c].value_counts(normalize=True)      # NaN is ignored here
    rare = share[share < RARE_THRESHOLD].index.tolist()
    
    if len(rare) >= 2 or (len(rare) == 1 and "Other" in share.index):
        rare_levels[c] = rare
    elif rare:
        print(f"{c}: only one rare level {rare} and no 'Other' -> left unchanged")

for c, levels in rare_levels.items():
    n_rows = train[c].isin(levels).sum()
    print(f"{c}: {len(levels)} rare levels, {n_rows} rows -> {levels}")

weather_pattern: only one rare level ['Typhoon'] and no 'Other' -> left unchanged
geological_era: 2 rare levels, 550 rows -> ['Cambrian', 'Permian']
region: 17 rare levels, 1499 rows -> ['South-Dakota', 'Oregon', 'Vermont', 'Kentucky', 'Rhode-Island', 'Wisconsin', 'Nevada', 'Idaho', 'Connecticut', 'Tennessee', 'Georgia', 'Virginia', 'Arkansas', 'Iowa', 'Missouri', 'West-Virginia', 'Nebraska']
mineral_type: 2 rare levels, 201 rows -> ['Flint', 'Chalk']


Before merging, check that the rare levels do not hide a strong signal
(positive rate = share of `yes`; overall it is about 0.24).

In [10]:
for c, levels in rare_levels.items():
    sub = train[train[c].isin(levels)]
    rate = sub.groupby(c)[TARGET].apply(lambda s: (s == "yes").mean()).round(3)
    cnt = sub[c].value_counts()
    print("\n==", c)
    print(pd.DataFrame({"count": cnt, "pos_rate": rate}).sort_values("count", ascending=False))


== geological_era
                count  pos_rate
geological_era                 
Cambrian          298     0.107
Permian           252     0.123

== region
               count  pos_rate
region                        
South-Dakota     178     0.219
Oregon           131     0.290
Vermont          128     0.383
Kentucky         123     0.114
Rhode-Island      95     0.074
Wisconsin         92     0.391
Nevada            82     0.256
Idaho             81     0.284
Connecticut       80     0.375
Tennessee         78     0.128
Georgia           76     0.197
Virginia          67     0.328
Arkansas          65     0.046
Iowa              61     0.049
Missouri          59     0.373
West-Virginia     54     0.222
Nebraska          49     0.020

== mineral_type
              count  pos_rate
mineral_type                 
Flint           154     0.032
Chalk            47     0.021


In [12]:
def group_rare(df, rare_levels):
    df = df.copy()
    for c, levels in rare_levels.items():
        df[c] = df[c].replace(levels, "Other")   # NaN stays NaN
    return df

train = group_rare(train, rare_levels)
test = group_rare(test, rare_levels)   # same levels (learned from train) applied to test

for c in rare_levels:
    print(f"{c}: levels {train_raw[c].nunique()} -> {train[c].nunique()}")

geological_era: levels 5 -> 4
region: levels 21 -> 4
mineral_type: levels 16 -> 15


In [14]:
# Positive rate of the new 'Other' level (train)
for c in rare_levels:
    other = train[train[c] == "Other"]
    print(f"{c:15s} Other: {len(other):5d} rows, pos_rate = {(other[TARGET] == 'yes').mean():.3f}")

geological_era  Other:   550 rows, pos_rate = 0.115
region          Other:  2030 rows, pos_rate = 0.223
mineral_type    Other:   201 rows, pos_rate = 0.030


### 2.3 Keep `Unknown` as its own category
`Unknown` in `species` / `personal_interest` is a hidden missing value, but Part 1 showed it is
**informative** (positive rate 8.7% vs 24% overall). Replacing it with the most frequent value would
throw that signal away, so it is kept unchanged. We only confirm it exists in both sets.

In [17]:
for c in ["species", "personal_interest"]:
    print(f"{c:18s} Unknown -> train: {(train[c] == 'Unknown').sum():5d} | test: {(test[c] == 'Unknown').sum():5d}")

species            Unknown -> train:  1760 | test:   767
personal_interest  Unknown -> train:  1767 | test:   770


### 2.4 Keep zeros, capped values and IQR outliers

In [20]:
both_zero = (train["performance_score"] == 0) & (train["stability_index"] == 0)
print("Double-zero rows kept (train):", both_zero.sum(),
      "| labels:", train.loc[both_zero, TARGET].value_counts().to_dict())

Double-zero rows kept (train): 265 | labels: {'no': 265}


### 2.5 Fill the remaining missing feature values

In [23]:
from sklearn.impute import SimpleImputer

print("NaN in features before -> train:", train[num_cols + cat_cols].isnull().sum().sum(),
      "| test:", test[num_cols + cat_cols].isnull().sum().sum())

num_imputer = SimpleImputer(strategy="median").fit(train[num_cols])
cat_imputer = SimpleImputer(strategy="most_frequent").fit(train[cat_cols])

print("\nLearned fill values (from train):")
print(pd.Series(num_imputer.statistics_, index=num_cols).round(3).to_string())
print(pd.Series(cat_imputer.statistics_, index=cat_cols).to_string())

NaN in features before -> train: 47 | test: 20

Learned fill values (from train):
aptitude_score       333.510
performance_score    275.390
stability_index        3.351
load_ratio            23.689
index_weight          11.730
activity_duration     80.330
composite_rank         3.199
geological_era        Triassic
weather_pattern        Monsoon
region                   Texas
personal_interest    Astronomy
brew_preference       Espresso
instrument               Cello
species                   Lynx
mineral_type            Marble


In [24]:
# Apply the SAME learned values to both sets
train_filled = train.copy()
test_filled = test.copy()
train_filled[num_cols] = num_imputer.transform(train[num_cols])
test_filled[num_cols] = num_imputer.transform(test[num_cols])
train_filled[cat_cols] = cat_imputer.transform(train[cat_cols])
test_filled[cat_cols] = cat_imputer.transform(test[cat_cols])

print("NaN in features after -> train:", train_filled[num_cols + cat_cols].isnull().sum().sum(),
      "| test:", test_filled[num_cols + cat_cols].isnull().sum().sum())

NaN in features after -> train: 0 | test: 0


### 2.6 Encode the label and summarise

In [28]:
train[TARGET] = (train[TARGET] == "yes").astype(int)
test[TARGET] = (test[TARGET] == "yes").astype(int)

print("Positive rate -> train:", round(train[TARGET].mean(), 3), "| test:", round(test[TARGET].mean(), 3))

Positive rate -> train: 0.24 | test: 0.238


In [30]:
summary = pd.DataFrame({
    "before": [len(train_raw), len(test_raw),
               train_raw[num_cols + cat_cols].isnull().any(axis=1).sum(),
               test_raw[num_cols + cat_cols].isnull().any(axis=1).sum(),
               sum(train_raw[c].nunique() for c in cat_cols)],
    "after": [len(train), len(test),
              train[num_cols + cat_cols].isnull().any(axis=1).sum(),
              test[num_cols + cat_cols].isnull().any(axis=1).sum(),
              sum(train[c].nunique() for c in cat_cols)],
}, index=["train rows", "test rows",
          "train rows with NaN feature (filled in Pipeline)",
          "test rows with NaN feature (filled in Pipeline)",
          "total categorical levels (train)"])
summary

,before,after
train rows,31112,31109
test rows,13334,13333
train rows with NaN feature (filled in Pipeline),47,47
test rows with NaN feature (filled in Pipeline),20,20
total categorical levels (train),75,56


In [32]:
train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)
print("Saved train_clean.csv", train.shape, "and test_clean.csv", test.shape)

Saved train_clean.csv (31109, 16) and test_clean.csv (13333, 16)
